In [1]:
import pandas as pd
import tarfile

# Processing original main dataset
### Descomprimir e abrir o dataset do main

In [2]:
df_react_rules = pd.read_csv('src/biocatalyzer/data/reactionrules/reaction_rules_biocatalyzer.tsv.bz2', compression = 'bz2', sep = '\t')
df_react_rules.head()

,InternalID,RuleID,Reactants,SMARTS,EC_Numbers,Organisms,ReactionRuleSource
0,Rule_0,"1,2-BenzoquinoneAdditionCyclization",Any,"[#7H2,SH1:11]-[#6:10]-[#6:9]-[#6:6]-1=[#6:1]-[...",spontaneous_reaction,spontaneous_reaction,ChemicalDamageMINE
1,Rule_1,"1,2-BenzoquinoneAlphaAddition_1,2-Benzoquinone",Any;O=C1C=CC=CC1=O,"[#6:10]-[#7H2,#16H1:9].[O:7]=[#6:3]-1-[#6;h1:4...",spontaneous_reaction,spontaneous_reaction,ChemicalDamageMINE
2,Rule_2,"1,2-BenzoquinoneAlphaAddition_Cystine",Any;N[C@@H](CS)C(=O)O,[O:7]=[#6:3]-1-[#6;h1:4]=[#6:5]-[#6:6]=[#6:1]-...,spontaneous_reaction,spontaneous_reaction,ChemicalDamageMINE
3,Rule_5,"1,2-BenzoquinoneBetaAddition_1,2Benzoquinone",Any;O=C1C=CC=CC1=O,"[#6:10]-[#7,#16;H1:9].[O:7]=[#6:3]-1-[#6:4]=[#...",spontaneous_reaction,spontaneous_reaction,ChemicalDamageMINE
4,Rule_6,"1,2-BenzoquinoneBetaAddition_Cystine",Any;N[C@@H](CS)C(=O)O,[O:7]=[#6:3]-1-[#6:4]=[#6;h1:5]-[#6:6]=[#6:1]-...,spontaneous_reaction,spontaneous_reaction,ChemicalDamageMINE


In [3]:
df_react_rules['ReactionRuleSource'].unique()

array(['ChemicalDamageMINE', 'EnzymaticMINE', 'KBPickaxe',
       'MetacycMINEIntermediate',
       'MetacycMINEIntermediate;MetacycMINEGeneralyzed',
       'MetacycMINEGeneralyzed', 'RetroRules'], dtype=object)

1º passo - Eliminar do dataset do main, todas as retrorules

In [4]:
df_react_rules_forward = df_react_rules[df_react_rules["ReactionRuleSource"] != "RetroRules"]
df_react_rules_forward.head()

,InternalID,RuleID,Reactants,SMARTS,EC_Numbers,Organisms,ReactionRuleSource
0,Rule_0,"1,2-BenzoquinoneAdditionCyclization",Any,"[#7H2,SH1:11]-[#6:10]-[#6:9]-[#6:6]-1=[#6:1]-[...",spontaneous_reaction,spontaneous_reaction,ChemicalDamageMINE
1,Rule_1,"1,2-BenzoquinoneAlphaAddition_1,2-Benzoquinone",Any;O=C1C=CC=CC1=O,"[#6:10]-[#7H2,#16H1:9].[O:7]=[#6:3]-1-[#6;h1:4...",spontaneous_reaction,spontaneous_reaction,ChemicalDamageMINE
2,Rule_2,"1,2-BenzoquinoneAlphaAddition_Cystine",Any;N[C@@H](CS)C(=O)O,[O:7]=[#6:3]-1-[#6;h1:4]=[#6:5]-[#6:6]=[#6:1]-...,spontaneous_reaction,spontaneous_reaction,ChemicalDamageMINE
3,Rule_5,"1,2-BenzoquinoneBetaAddition_1,2Benzoquinone",Any;O=C1C=CC=CC1=O,"[#6:10]-[#7,#16;H1:9].[O:7]=[#6:3]-1-[#6:4]=[#...",spontaneous_reaction,spontaneous_reaction,ChemicalDamageMINE
4,Rule_6,"1,2-BenzoquinoneBetaAddition_Cystine",Any;N[C@@H](CS)C(=O)O,[O:7]=[#6:3]-1-[#6:4]=[#6;h1:5]-[#6:6]=[#6:1]-...,spontaneous_reaction,spontaneous_reaction,ChemicalDamageMINE


In [5]:
df_react_rules_forward['ReactionRuleSource'].unique()

array(['ChemicalDamageMINE', 'EnzymaticMINE', 'KBPickaxe',
       'MetacycMINEIntermediate',
       'MetacycMINEIntermediate;MetacycMINEGeneralyzed',
       'MetacycMINEGeneralyzed'], dtype=object)

In [6]:
df_react_rules_forward['EC_Numbers'].unique()

array(['spontaneous_reaction', nan, '1.14.11.32', ..., '1.14.16.3',
       '2.4.2.31', '1.3.1.63'], dtype=object)

In [7]:
df_react_rules_forward['SMARTS'].unique()

array(['[#7H2,SH1:11]-[#6:10]-[#6:9]-[#6:6]-1=[#6:1]-[#6:2](=[O:8])-[#6:3](=[O:7])-[#6:4]=[#6;h1:5]-1>>[#8:8]-[#6:2]-1=[#6:3](-[#8:7])-[#6:4]=[#6:5]-2-[*:11]-[#6:10]-[#6:9]-[#6:6]-2=[#6:1]-1',
       '[#6:10]-[#7H2,#16H1:9].[O:7]=[#6:3]-1-[#6;h1:4]=[#6:5]-[#6:6]=[#6:1]-[#6:2]-1=[O:8]>>[#6:10]-[*:9]-[#6:4]-1=[#6:5]-[#6:6]=[#6:1]-[#6:2](-[#8:8])=[#6:3]-1-[#8:7]',
       '[O:7]=[#6:3]-1-[#6;h1:4]=[#6:5]-[#6:6]=[#6:1]-[#6:2]-1=[O:8].[#6:10]-[#7H2,#16H1:9]>>[#6:10]-[*:9]-[#6:4]-1=[#6:5]-[#6:6]=[#6:1]-[#6:2](-[#8:8])=[#6:3]-1-[#8:7]',
       ...,
       '[#7:1]-[#6:2]-[#6:3]-[#8:4].[#8:5].[#8:6]=[#8:7]>>[#6:2]=[#8:5].[#6:3]=[#8:4].[#7:1].[#8:6]-[#8:7]',
       '[#6:1]-[#8:2].[#6:3]=[#8:4].[#6:5]=[#8:6].[#8:7]-[#16:8]>>[#6:3]-[#16:8]=[#8:7].[#6:1]-[#6:5]-[#8:6].[#8:2]=[#8:4]',
       '[#6:1]-[#16:2]=[#8:3].[#6:4]-[#6:5]-[#8:6].[#8:7]=[#8:8]>>[#6:4]-[#8:7].[#6:1]=[#8:8].[#6:5]=[#8:6].[#8:3]-[#16:2]'],
      dtype=object)

In [8]:
df_react_rules_forward

,InternalID,RuleID,Reactants,SMARTS,EC_Numbers,Organisms,ReactionRuleSource
0,Rule_0,"1,2-BenzoquinoneAdditionCyclization",Any,"[#7H2,SH1:11]-[#6:10]-[#6:9]-[#6:6]-1=[#6:1]-[...",spontaneous_reaction,spontaneous_reaction,ChemicalDamageMINE
1,Rule_1,"1,2-BenzoquinoneAlphaAddition_1,2-Benzoquinone",Any;O=C1C=CC=CC1=O,"[#6:10]-[#7H2,#16H1:9].[O:7]=[#6:3]-1-[#6;h1:4...",spontaneous_reaction,spontaneous_reaction,ChemicalDamageMINE
2,Rule_2,"1,2-BenzoquinoneAlphaAddition_Cystine",Any;N[C@@H](CS)C(=O)O,[O:7]=[#6:3]-1-[#6;h1:4]=[#6:5]-[#6:6]=[#6:1]-...,spontaneous_reaction,spontaneous_reaction,ChemicalDamageMINE
3,Rule_5,"1,2-BenzoquinoneBetaAddition_1,2Benzoquinone",Any;O=C1C=CC=CC1=O,"[#6:10]-[#7,#16;H1:9].[O:7]=[#6:3]-1-[#6:4]=[#...",spontaneous_reaction,spontaneous_reaction,ChemicalDamageMINE
4,Rule_6,"1,2-BenzoquinoneBetaAddition_Cystine",Any;N[C@@H](CS)C(=O)O,[O:7]=[#6:3]-1-[#6:4]=[#6;h1:5]-[#6:6]=[#6:1]-...,spontaneous_reaction,spontaneous_reaction,ChemicalDamageMINE
...,...,...,...,...,...,...,...
9909,Rule_47371,rule1927,Any;Any;O=O,[#6:1]-[#6:2]-[#8:3].[#6:4]-[#7:5]:[#6:6]=[#7:...,NaN,NaN,MetacycMINEGeneralyzed
9910,Rule_47396,rule2157,Any;O=C=O;N;OO,[#6:1]=[#8:2].[#6:3]=[#8:4].[#7:5].[#8:6]-[#8:...,NaN,NaN,MetacycMINEGeneralyzed
9911,Rule_47397,rule2158,Any;O;O=O,[#7:1]-[#6:2]-[#6:3]-[#8:4].[#8:5].[#8:6]=[#8:...,NaN,NaN,MetacycMINEGeneralyzed
9912,Rule_47410,rule2233,Any;Any;O=C=O;O=S(O)O,[#6:1]-[#8:2].[#6:3]=[#8:4].[#6:5]=[#8:6].[#8:...,NaN,NaN,MetacycMINEGeneralyzed


# Processing RetroRules Dataset 
### Depois de sacar do RetroRules, o ficheiro "retrorules_rr02_rp3_hs.tar.gz"

O ficheiro retrorules_rr02_rp3_hs.tar.gz contém os seguintes ficheiros:

    retrorules_rr02_flat_all.tsv – Um ficheiro de texto tabulado (.tsv), que provavelmente contém dados estruturados, possivelmente relacionados com regras retroquímicas.
    README.md – Um ficheiro de documentação (Markdown), que pode conter informações sobre o conteúdo do ficheiro e instruções sobre a sua utilização.
    ._retrorules_rr02_flat_all.tsv – Um ficheiro auxiliar, possivelmente criado automaticamente pelo sistema.

In [9]:
with tarfile.open("../data/retrorules_rr02_rp3_hs.tar.gz", "r:gz") as tar:
    extracted_file = tar.extractfile("retrorules_rr02_rp3_hs/retrorules_rr02_flat_all.tsv")
    
    df_retro = pd.read_csv(extracted_file, sep="\t")

df_retro.head()

,# Rule_ID,Legacy_ID,Reaction_ID,Diameter,Rule_order,Rule_SMARTS,Substrate_ID,Substrate_SMILES,Product_IDs,Product_SMILES,Rule_SMILES,Rule_SMARTS_lite,Score,Score_normalized,Reaction_EC_number,Reaction_direction,Rule_relative_direction,Rule_usage
0,RR-02-fbdda75e23f518b6-02-F,MNXR94682_MNXM821,MNXR94682,2,1,([#6&v4:1](=[#8&v2:2])(-[#6&v4:3](-[#6&v4:4])(...,MNXM821,[H][C](=[O])[C]([H])([H])[C]([H])([H])[H],MNXM90191,[H][O][C]([H])([H])[C]([H])([O][H])[C]([H])([H...,[C](=[O])(-[C](-[C])(-[H])-[H])-[H]>>[C](-[O]-...,([#6&v4](=[#8&v2])(-[#6&v4](-[#6&v4])(-[#1&v1]...,4.295611,0.104697,NaN,0,1,both
1,RR-02-0250d458c4991a7d-02-F,MNXR94682_MNXM90191,MNXR94682,2,1,([#6&v4:1](-[#8&v2:2]-[#1&v1:3])(-[#6&v4:4](-[...,MNXM90191,[H][O][C]([H])([H])[C]([H])([O][H])[C]([H])([H...,MNXM2.MNXM821,[H][O][H].[H][C](=[O])[C]([H])([H])[C]([H])([H...,[C](-[O]-[H])(-[C](-[C])(-[O]-[H])-[H])(-[H])-...,([#6&v4](-[#8&v2]-[#1&v1])(-[#6&v4](-[#6&v4])(...,4.295611,0.104697,NaN,0,-1,both
2,RR-02-c3681aa8011dc014-02-F,MNXR94689_MNXM101404,MNXR94689,2,1,([#6&v4:1]-[#6&v4:2](=[#8&v2:3])-[#6&v4:4])>>(...,MNXM101404,[H][O][C](=[O])[C]([H])([H])[C]([H])([H])[C]([...,MNXM9689,[H][O][C](=[O])[C]([H])([H])[C]([H])([H])[C]([...,[C]-[C](=[O])-[C]>>[C]-[C](-[O]-[H])(-[C])-[H],([#6&v4]-[#6&v4](=[#8&v2])-[#6&v4])>>([#6&v4]-...,3.068557,0.140673,1.1.1.-,0,1,both
3,RR-02-1364a3f2a297c78c-02-F,MNXR94689_MNXM9689,MNXR94689,2,1,([#6&v4:1]-[#6&v4:2](-[#8&v2:3]-[#1&v1:4])(-[#...,MNXM9689,[H][O][C](=[O])[C]([H])([H])[C]([H])([H])[C]([...,MNXM1.MNXM10.MNXM101404,[H+].[H][N]=[C]([O][H])[C]1=[C]([H])[N]([C]2([...,[C]-[C](-[O]-[H])(-[C])-[H]>>[C]-[C](=[O])-[C]...,([#6&v4]-[#6&v4](-[#8&v2]-[#1&v1])(-[#6&v4])-[...,2.875061,0.148733,1.1.1.-,0,-1,both
4,RR-02-2860703b5bba4808-02-F,MNXR94690_MNXM2313,MNXR94690,2,1,([#8&v2:1](-[#8&v2:2]-[#6&v4:3])-[#1&v1:4])>>(...,MNXM2313,[H][O][O][C]([H])([C]([H])=[C]([H])[C]([H])=[C...,MNXM2.MNXM9689,[H][O][H].[H][O][C](=[O])[C]([H])([H])[C]([H])...,[O](-[O]-[C])-[H]>>[O](-[C])-[H].[O](-[H])-[H],([#8&v2](-[#8&v2]-[#6&v4])-[#1&v1])>>([#8&v2](...,0.000000,1.000000,NaN,0,1,both


Processamento das colunas do dataset retrorules_rr02_rp3_hs.tar.gz, para depois adicionar ao dataset reaction_rules_biocatalyzer.tsv.bz2, preprocessado sem as retrorules.

### Eliminar retro rules
Deixamos estar both e forward

In [10]:
# Mostrar os elementos da coluna 'Rule_usage'
df_retro['Rule_usage'].unique()

array(['both', 'forward', 'retro'], dtype=object)

In [11]:
df_forward = df_retro[df_retro["Rule_usage"] != "retro"]
df_forward.head()

,# Rule_ID,Legacy_ID,Reaction_ID,Diameter,Rule_order,Rule_SMARTS,Substrate_ID,Substrate_SMILES,Product_IDs,Product_SMILES,Rule_SMILES,Rule_SMARTS_lite,Score,Score_normalized,Reaction_EC_number,Reaction_direction,Rule_relative_direction,Rule_usage
0,RR-02-fbdda75e23f518b6-02-F,MNXR94682_MNXM821,MNXR94682,2,1,([#6&v4:1](=[#8&v2:2])(-[#6&v4:3](-[#6&v4:4])(...,MNXM821,[H][C](=[O])[C]([H])([H])[C]([H])([H])[H],MNXM90191,[H][O][C]([H])([H])[C]([H])([O][H])[C]([H])([H...,[C](=[O])(-[C](-[C])(-[H])-[H])-[H]>>[C](-[O]-...,([#6&v4](=[#8&v2])(-[#6&v4](-[#6&v4])(-[#1&v1]...,4.295611,0.104697,NaN,0,1,both
1,RR-02-0250d458c4991a7d-02-F,MNXR94682_MNXM90191,MNXR94682,2,1,([#6&v4:1](-[#8&v2:2]-[#1&v1:3])(-[#6&v4:4](-[...,MNXM90191,[H][O][C]([H])([H])[C]([H])([O][H])[C]([H])([H...,MNXM2.MNXM821,[H][O][H].[H][C](=[O])[C]([H])([H])[C]([H])([H...,[C](-[O]-[H])(-[C](-[C])(-[O]-[H])-[H])(-[H])-...,([#6&v4](-[#8&v2]-[#1&v1])(-[#6&v4](-[#6&v4])(...,4.295611,0.104697,NaN,0,-1,both
2,RR-02-c3681aa8011dc014-02-F,MNXR94689_MNXM101404,MNXR94689,2,1,([#6&v4:1]-[#6&v4:2](=[#8&v2:3])-[#6&v4:4])>>(...,MNXM101404,[H][O][C](=[O])[C]([H])([H])[C]([H])([H])[C]([...,MNXM9689,[H][O][C](=[O])[C]([H])([H])[C]([H])([H])[C]([...,[C]-[C](=[O])-[C]>>[C]-[C](-[O]-[H])(-[C])-[H],([#6&v4]-[#6&v4](=[#8&v2])-[#6&v4])>>([#6&v4]-...,3.068557,0.140673,1.1.1.-,0,1,both
3,RR-02-1364a3f2a297c78c-02-F,MNXR94689_MNXM9689,MNXR94689,2,1,([#6&v4:1]-[#6&v4:2](-[#8&v2:3]-[#1&v1:4])(-[#...,MNXM9689,[H][O][C](=[O])[C]([H])([H])[C]([H])([H])[C]([...,MNXM1.MNXM10.MNXM101404,[H+].[H][N]=[C]([O][H])[C]1=[C]([H])[N]([C]2([...,[C]-[C](-[O]-[H])(-[C])-[H]>>[C]-[C](=[O])-[C]...,([#6&v4]-[#6&v4](-[#8&v2]-[#1&v1])(-[#6&v4])-[...,2.875061,0.148733,1.1.1.-,0,-1,both
4,RR-02-2860703b5bba4808-02-F,MNXR94690_MNXM2313,MNXR94690,2,1,([#8&v2:1](-[#8&v2:2]-[#6&v4:3])-[#1&v1:4])>>(...,MNXM2313,[H][O][O][C]([H])([C]([H])=[C]([H])[C]([H])=[C...,MNXM2.MNXM9689,[H][O][H].[H][O][C](=[O])[C]([H])([H])[C]([H])...,[O](-[O]-[C])-[H]>>[O](-[C])-[H].[O](-[H])-[H],([#8&v2](-[#8&v2]-[#6&v4])-[#1&v1])>>([#8&v2](...,0.000000,1.000000,NaN,0,1,both


In [12]:
df_forward['Rule_usage'].unique()

array(['both', 'forward'], dtype=object)

### Renomear "# Rule_ID" para "RuleID"

In [13]:
df_forward1 = df_forward.rename(columns={"# Rule_ID": "RuleID"})
df_forward1.head()

,RuleID,Legacy_ID,Reaction_ID,Diameter,Rule_order,Rule_SMARTS,Substrate_ID,Substrate_SMILES,Product_IDs,Product_SMILES,Rule_SMILES,Rule_SMARTS_lite,Score,Score_normalized,Reaction_EC_number,Reaction_direction,Rule_relative_direction,Rule_usage
0,RR-02-fbdda75e23f518b6-02-F,MNXR94682_MNXM821,MNXR94682,2,1,([#6&v4:1](=[#8&v2:2])(-[#6&v4:3](-[#6&v4:4])(...,MNXM821,[H][C](=[O])[C]([H])([H])[C]([H])([H])[H],MNXM90191,[H][O][C]([H])([H])[C]([H])([O][H])[C]([H])([H...,[C](=[O])(-[C](-[C])(-[H])-[H])-[H]>>[C](-[O]-...,([#6&v4](=[#8&v2])(-[#6&v4](-[#6&v4])(-[#1&v1]...,4.295611,0.104697,NaN,0,1,both
1,RR-02-0250d458c4991a7d-02-F,MNXR94682_MNXM90191,MNXR94682,2,1,([#6&v4:1](-[#8&v2:2]-[#1&v1:3])(-[#6&v4:4](-[...,MNXM90191,[H][O][C]([H])([H])[C]([H])([O][H])[C]([H])([H...,MNXM2.MNXM821,[H][O][H].[H][C](=[O])[C]([H])([H])[C]([H])([H...,[C](-[O]-[H])(-[C](-[C])(-[O]-[H])-[H])(-[H])-...,([#6&v4](-[#8&v2]-[#1&v1])(-[#6&v4](-[#6&v4])(...,4.295611,0.104697,NaN,0,-1,both
2,RR-02-c3681aa8011dc014-02-F,MNXR94689_MNXM101404,MNXR94689,2,1,([#6&v4:1]-[#6&v4:2](=[#8&v2:3])-[#6&v4:4])>>(...,MNXM101404,[H][O][C](=[O])[C]([H])([H])[C]([H])([H])[C]([...,MNXM9689,[H][O][C](=[O])[C]([H])([H])[C]([H])([H])[C]([...,[C]-[C](=[O])-[C]>>[C]-[C](-[O]-[H])(-[C])-[H],([#6&v4]-[#6&v4](=[#8&v2])-[#6&v4])>>([#6&v4]-...,3.068557,0.140673,1.1.1.-,0,1,both
3,RR-02-1364a3f2a297c78c-02-F,MNXR94689_MNXM9689,MNXR94689,2,1,([#6&v4:1]-[#6&v4:2](-[#8&v2:3]-[#1&v1:4])(-[#...,MNXM9689,[H][O][C](=[O])[C]([H])([H])[C]([H])([H])[C]([...,MNXM1.MNXM10.MNXM101404,[H+].[H][N]=[C]([O][H])[C]1=[C]([H])[N]([C]2([...,[C]-[C](-[O]-[H])(-[C])-[H]>>[C]-[C](=[O])-[C]...,([#6&v4]-[#6&v4](-[#8&v2]-[#1&v1])(-[#6&v4])-[...,2.875061,0.148733,1.1.1.-,0,-1,both
4,RR-02-2860703b5bba4808-02-F,MNXR94690_MNXM2313,MNXR94690,2,1,([#8&v2:1](-[#8&v2:2]-[#6&v4:3])-[#1&v1:4])>>(...,MNXM2313,[H][O][O][C]([H])([C]([H])=[C]([H])[C]([H])=[C...,MNXM2.MNXM9689,[H][O][H].[H][O][C](=[O])[C]([H])([H])[C]([H])...,[O](-[O]-[C])-[H]>>[O](-[C])-[H].[O](-[H])-[H],([#8&v2](-[#8&v2]-[#6&v4])-[#1&v1])>>([#8&v2](...,0.000000,1.000000,NaN,0,1,both


### Renomear "Reaction_EC_number" para "EC_Numbers"

In [14]:
df_forward1['Reaction_EC_number'].unique()

array([nan, '1.1.1.-', '1.1.1.202', ..., '1.1.5.2', '2.7.7.63',
       '2.4.1.17,3.2.1.-'], dtype=object)

In [15]:
df_forward2 = df_forward1.rename(columns={"Reaction_EC_number": "EC_Numbers"})
df_forward2.head()

,RuleID,Legacy_ID,Reaction_ID,Diameter,Rule_order,Rule_SMARTS,Substrate_ID,Substrate_SMILES,Product_IDs,Product_SMILES,Rule_SMILES,Rule_SMARTS_lite,Score,Score_normalized,EC_Numbers,Reaction_direction,Rule_relative_direction,Rule_usage
0,RR-02-fbdda75e23f518b6-02-F,MNXR94682_MNXM821,MNXR94682,2,1,([#6&v4:1](=[#8&v2:2])(-[#6&v4:3](-[#6&v4:4])(...,MNXM821,[H][C](=[O])[C]([H])([H])[C]([H])([H])[H],MNXM90191,[H][O][C]([H])([H])[C]([H])([O][H])[C]([H])([H...,[C](=[O])(-[C](-[C])(-[H])-[H])-[H]>>[C](-[O]-...,([#6&v4](=[#8&v2])(-[#6&v4](-[#6&v4])(-[#1&v1]...,4.295611,0.104697,NaN,0,1,both
1,RR-02-0250d458c4991a7d-02-F,MNXR94682_MNXM90191,MNXR94682,2,1,([#6&v4:1](-[#8&v2:2]-[#1&v1:3])(-[#6&v4:4](-[...,MNXM90191,[H][O][C]([H])([H])[C]([H])([O][H])[C]([H])([H...,MNXM2.MNXM821,[H][O][H].[H][C](=[O])[C]([H])([H])[C]([H])([H...,[C](-[O]-[H])(-[C](-[C])(-[O]-[H])-[H])(-[H])-...,([#6&v4](-[#8&v2]-[#1&v1])(-[#6&v4](-[#6&v4])(...,4.295611,0.104697,NaN,0,-1,both
2,RR-02-c3681aa8011dc014-02-F,MNXR94689_MNXM101404,MNXR94689,2,1,([#6&v4:1]-[#6&v4:2](=[#8&v2:3])-[#6&v4:4])>>(...,MNXM101404,[H][O][C](=[O])[C]([H])([H])[C]([H])([H])[C]([...,MNXM9689,[H][O][C](=[O])[C]([H])([H])[C]([H])([H])[C]([...,[C]-[C](=[O])-[C]>>[C]-[C](-[O]-[H])(-[C])-[H],([#6&v4]-[#6&v4](=[#8&v2])-[#6&v4])>>([#6&v4]-...,3.068557,0.140673,1.1.1.-,0,1,both
3,RR-02-1364a3f2a297c78c-02-F,MNXR94689_MNXM9689,MNXR94689,2,1,([#6&v4:1]-[#6&v4:2](-[#8&v2:3]-[#1&v1:4])(-[#...,MNXM9689,[H][O][C](=[O])[C]([H])([H])[C]([H])([H])[C]([...,MNXM1.MNXM10.MNXM101404,[H+].[H][N]=[C]([O][H])[C]1=[C]([H])[N]([C]2([...,[C]-[C](-[O]-[H])(-[C])-[H]>>[C]-[C](=[O])-[C]...,([#6&v4]-[#6&v4](-[#8&v2]-[#1&v1])(-[#6&v4])-[...,2.875061,0.148733,1.1.1.-,0,-1,both
4,RR-02-2860703b5bba4808-02-F,MNXR94690_MNXM2313,MNXR94690,2,1,([#8&v2:1](-[#8&v2:2]-[#6&v4:3])-[#1&v1:4])>>(...,MNXM2313,[H][O][O][C]([H])([C]([H])=[C]([H])[C]([H])=[C...,MNXM2.MNXM9689,[H][O][H].[H][O][C](=[O])[C]([H])([H])[C]([H])...,[O](-[O]-[C])-[H]>>[O](-[C])-[H].[O](-[H])-[H],([#8&v2](-[#8&v2]-[#6&v4])-[#1&v1])>>([#8&v2](...,0.000000,1.000000,NaN,0,1,both


### Renomear "Rule_SMARTS" para "SMARTS"

In [16]:
df_forward2['Rule_SMARTS'].unique()

array(['([#6&v4:1](=[#8&v2:2])(-[#6&v4:3](-[#6&v4:4])(-[#1&v1:5])-[#1&v1:6])-[#1&v1:7])>>([#6&v4:1](-[#8&v2:2]-[#1&v1:6])(-[#6&v4:3](-[#6&v4:4])(-[#8&v2]-[#1&v1])-[#1&v1:5])(-[#1&v1:7])-[#1&v1])',
       '([#6&v4:1](-[#8&v2:2]-[#1&v1:3])(-[#6&v4:4](-[#6&v4:5])(-[#8&v2:6]-[#1&v1:7])-[#1&v1:8])(-[#1&v1:9])-[#1&v1:10])>>([#6&v4:1](=[#8&v2:2])(-[#6&v4:4](-[#6&v4:5])(-[#1&v1:8])-[#1&v1:3])-[#1&v1:9].[#8&v2:6](-[#1&v1:7])-[#1&v1:10])',
       '([#6&v4:1]-[#6&v4:2](=[#8&v2:3])-[#6&v4:4])>>([#6&v4:1]-[#6&v4:2](-[#8&v2:3]-[#1&v1])(-[#6&v4:4])-[#1&v1])',
       ...,
       '([#15&v5:1](=[#8&v2:2])(-[#8&v2:3]-[#6&v4:4]1(-[#1&v1:5])-[#6&v4:6](-[#8&v2:7]-[#1&v1:8])(-[#1&v1:9])-[#6&v4:10](-[#8&v2:11]-[#1&v1:12])(-[#1&v1:13])-[#6&v4:14](-[#8&v2:15]-[#1&v1:16])(-[#1&v1:17])-[#6&v4:18](-[#6&v4:19](-[#8&v2:20]-[#1&v1:21])=[#8&v2:22])(-[#1&v1:23])-[#8&v2:24]-1)(-[#8&v2:25]-[#1&v1:26])-[#8&v2:27]-[#15&v5:28](-[#8&v2:29]-[#6&v4:30](-[#6&v4:31]1(-[#1&v1:32])-[#6&v4:33](-[#8&v2:34])(-[#1&v1:35])-[#6&v4:36]-[

In [17]:
df_forward3 = df_forward2.rename(columns={"Rule_SMARTS": "SMARTS"})
df_forward3.head()

,RuleID,Legacy_ID,Reaction_ID,Diameter,Rule_order,SMARTS,Substrate_ID,Substrate_SMILES,Product_IDs,Product_SMILES,Rule_SMILES,Rule_SMARTS_lite,Score,Score_normalized,EC_Numbers,Reaction_direction,Rule_relative_direction,Rule_usage
0,RR-02-fbdda75e23f518b6-02-F,MNXR94682_MNXM821,MNXR94682,2,1,([#6&v4:1](=[#8&v2:2])(-[#6&v4:3](-[#6&v4:4])(...,MNXM821,[H][C](=[O])[C]([H])([H])[C]([H])([H])[H],MNXM90191,[H][O][C]([H])([H])[C]([H])([O][H])[C]([H])([H...,[C](=[O])(-[C](-[C])(-[H])-[H])-[H]>>[C](-[O]-...,([#6&v4](=[#8&v2])(-[#6&v4](-[#6&v4])(-[#1&v1]...,4.295611,0.104697,NaN,0,1,both
1,RR-02-0250d458c4991a7d-02-F,MNXR94682_MNXM90191,MNXR94682,2,1,([#6&v4:1](-[#8&v2:2]-[#1&v1:3])(-[#6&v4:4](-[...,MNXM90191,[H][O][C]([H])([H])[C]([H])([O][H])[C]([H])([H...,MNXM2.MNXM821,[H][O][H].[H][C](=[O])[C]([H])([H])[C]([H])([H...,[C](-[O]-[H])(-[C](-[C])(-[O]-[H])-[H])(-[H])-...,([#6&v4](-[#8&v2]-[#1&v1])(-[#6&v4](-[#6&v4])(...,4.295611,0.104697,NaN,0,-1,both
2,RR-02-c3681aa8011dc014-02-F,MNXR94689_MNXM101404,MNXR94689,2,1,([#6&v4:1]-[#6&v4:2](=[#8&v2:3])-[#6&v4:4])>>(...,MNXM101404,[H][O][C](=[O])[C]([H])([H])[C]([H])([H])[C]([...,MNXM9689,[H][O][C](=[O])[C]([H])([H])[C]([H])([H])[C]([...,[C]-[C](=[O])-[C]>>[C]-[C](-[O]-[H])(-[C])-[H],([#6&v4]-[#6&v4](=[#8&v2])-[#6&v4])>>([#6&v4]-...,3.068557,0.140673,1.1.1.-,0,1,both
3,RR-02-1364a3f2a297c78c-02-F,MNXR94689_MNXM9689,MNXR94689,2,1,([#6&v4:1]-[#6&v4:2](-[#8&v2:3]-[#1&v1:4])(-[#...,MNXM9689,[H][O][C](=[O])[C]([H])([H])[C]([H])([H])[C]([...,MNXM1.MNXM10.MNXM101404,[H+].[H][N]=[C]([O][H])[C]1=[C]([H])[N]([C]2([...,[C]-[C](-[O]-[H])(-[C])-[H]>>[C]-[C](=[O])-[C]...,([#6&v4]-[#6&v4](-[#8&v2]-[#1&v1])(-[#6&v4])-[...,2.875061,0.148733,1.1.1.-,0,-1,both
4,RR-02-2860703b5bba4808-02-F,MNXR94690_MNXM2313,MNXR94690,2,1,([#8&v2:1](-[#8&v2:2]-[#6&v4:3])-[#1&v1:4])>>(...,MNXM2313,[H][O][O][C]([H])([C]([H])=[C]([H])[C]([H])=[C...,MNXM2.MNXM9689,[H][O][H].[H][O][C](=[O])[C]([H])([H])[C]([H])...,[O](-[O]-[C])-[H]>>[O](-[C])-[H].[O](-[H])-[H],([#8&v2](-[#8&v2]-[#6&v4])-[#1&v1])>>([#8&v2](...,0.000000,1.000000,NaN,0,1,both


### Juntar as colunas "RuleID" e "Legacy_ID" -> "RuleID_Legacy_ID" e renomear para "RuleID"

In [18]:
df_rr_ruleid = df_forward3.copy()
df_rr_ruleid['RuleID'] = df_rr_ruleid['RuleID'] + '_' + df_rr_ruleid['Legacy_ID']
df_rr_ruleid = df_rr_ruleid.drop(columns=['Legacy_ID'])
df_rr_ruleid.head()

,RuleID,Reaction_ID,Diameter,Rule_order,SMARTS,Substrate_ID,Substrate_SMILES,Product_IDs,Product_SMILES,Rule_SMILES,Rule_SMARTS_lite,Score,Score_normalized,EC_Numbers,Reaction_direction,Rule_relative_direction,Rule_usage
0,RR-02-fbdda75e23f518b6-02-F_MNXR94682_MNXM821,MNXR94682,2,1,([#6&v4:1](=[#8&v2:2])(-[#6&v4:3](-[#6&v4:4])(...,MNXM821,[H][C](=[O])[C]([H])([H])[C]([H])([H])[H],MNXM90191,[H][O][C]([H])([H])[C]([H])([O][H])[C]([H])([H...,[C](=[O])(-[C](-[C])(-[H])-[H])-[H]>>[C](-[O]-...,([#6&v4](=[#8&v2])(-[#6&v4](-[#6&v4])(-[#1&v1]...,4.295611,0.104697,NaN,0,1,both
1,RR-02-0250d458c4991a7d-02-F_MNXR94682_MNXM90191,MNXR94682,2,1,([#6&v4:1](-[#8&v2:2]-[#1&v1:3])(-[#6&v4:4](-[...,MNXM90191,[H][O][C]([H])([H])[C]([H])([O][H])[C]([H])([H...,MNXM2.MNXM821,[H][O][H].[H][C](=[O])[C]([H])([H])[C]([H])([H...,[C](-[O]-[H])(-[C](-[C])(-[O]-[H])-[H])(-[H])-...,([#6&v4](-[#8&v2]-[#1&v1])(-[#6&v4](-[#6&v4])(...,4.295611,0.104697,NaN,0,-1,both
2,RR-02-c3681aa8011dc014-02-F_MNXR94689_MNXM101404,MNXR94689,2,1,([#6&v4:1]-[#6&v4:2](=[#8&v2:3])-[#6&v4:4])>>(...,MNXM101404,[H][O][C](=[O])[C]([H])([H])[C]([H])([H])[C]([...,MNXM9689,[H][O][C](=[O])[C]([H])([H])[C]([H])([H])[C]([...,[C]-[C](=[O])-[C]>>[C]-[C](-[O]-[H])(-[C])-[H],([#6&v4]-[#6&v4](=[#8&v2])-[#6&v4])>>([#6&v4]-...,3.068557,0.140673,1.1.1.-,0,1,both
3,RR-02-1364a3f2a297c78c-02-F_MNXR94689_MNXM9689,MNXR94689,2,1,([#6&v4:1]-[#6&v4:2](-[#8&v2:3]-[#1&v1:4])(-[#...,MNXM9689,[H][O][C](=[O])[C]([H])([H])[C]([H])([H])[C]([...,MNXM1.MNXM10.MNXM101404,[H+].[H][N]=[C]([O][H])[C]1=[C]([H])[N]([C]2([...,[C]-[C](-[O]-[H])(-[C])-[H]>>[C]-[C](=[O])-[C]...,([#6&v4]-[#6&v4](-[#8&v2]-[#1&v1])(-[#6&v4])-[...,2.875061,0.148733,1.1.1.-,0,-1,both
4,RR-02-2860703b5bba4808-02-F_MNXR94690_MNXM2313,MNXR94690,2,1,([#8&v2:1](-[#8&v2:2]-[#6&v4:3])-[#1&v1:4])>>(...,MNXM2313,[H][O][O][C]([H])([C]([H])=[C]([H])[C]([H])=[C...,MNXM2.MNXM9689,[H][O][H].[H][O][C](=[O])[C]([H])([H])[C]([H])...,[O](-[O]-[C])-[H]>>[O](-[C])-[H].[O](-[H])-[H],([#8&v2](-[#8&v2]-[#6&v4])-[#1&v1])>>([#8&v2](...,0.000000,1.000000,NaN,0,1,both


# Merge two datasets: Original Main dataset with RetroReules dataset

In [19]:
df_rr_ruleid.columns.tolist()

['RuleID',
 'Reaction_ID',
 'Diameter',
 'Rule_order',
 'SMARTS',
 'Substrate_ID',
 'Substrate_SMILES',
 'Product_IDs',
 'Product_SMILES',
 'Rule_SMILES',
 'Rule_SMARTS_lite',
 'Score',
 'Score_normalized',
 'EC_Numbers',
 'Reaction_direction',
 'Rule_relative_direction',
 'Rule_usage']

In [20]:
df_react_rules_forward.columns.tolist()

['InternalID',
 'RuleID',
 'Reactants',
 'SMARTS',
 'EC_Numbers',
 'Organisms',
 'ReactionRuleSource']

In [21]:
df_react_rules_forward['Diameter'] = 'None'

C:\Users\Duarte Velho\AppData\Local\Temp\ipykernel_2088\4025000396.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_react_rules_forward['Diameter'] = 'None'


In [22]:
df_rr_cut = df_rr_ruleid[['RuleID', 'EC_Numbers', 'SMARTS', 'Diameter']]
df_rr_cut.head()

,RuleID,EC_Numbers,SMARTS,Diameter
0,RR-02-fbdda75e23f518b6-02-F_MNXR94682_MNXM821,NaN,([#6&v4:1](=[#8&v2:2])(-[#6&v4:3](-[#6&v4:4])(...,2
1,RR-02-0250d458c4991a7d-02-F_MNXR94682_MNXM90191,NaN,([#6&v4:1](-[#8&v2:2]-[#1&v1:3])(-[#6&v4:4](-[...,2
2,RR-02-c3681aa8011dc014-02-F_MNXR94689_MNXM101404,1.1.1.-,([#6&v4:1]-[#6&v4:2](=[#8&v2:3])-[#6&v4:4])>>(...,2
3,RR-02-1364a3f2a297c78c-02-F_MNXR94689_MNXM9689,1.1.1.-,([#6&v4:1]-[#6&v4:2](-[#8&v2:3]-[#1&v1:4])(-[#...,2
4,RR-02-2860703b5bba4808-02-F_MNXR94690_MNXM2313,NaN,([#8&v2:1](-[#8&v2:2]-[#6&v4:3])-[#1&v1:4])>>(...,2


In [23]:
max_internal_id = df_react_rules_forward['InternalID'].str.extract('(\d+)').astype(int).max()[0]
df_rr_cut['InternalID'] = ['Rule_' + str(i) for i in range(max_internal_id + 1, max_internal_id + 1 + len(df_rr_cut))]
df_rr_cut.head()

C:\Users\Duarte Velho\AppData\Local\Temp\ipykernel_2088\985051467.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_rr_cut['InternalID'] = ['Rule_' + str(i) for i in range(max_internal_id + 1, max_internal_id + 1 + len(df_rr_cut))]


,RuleID,EC_Numbers,SMARTS,Diameter,InternalID
0,RR-02-fbdda75e23f518b6-02-F_MNXR94682_MNXM821,NaN,([#6&v4:1](=[#8&v2:2])(-[#6&v4:3](-[#6&v4:4])(...,2,Rule_47412
1,RR-02-0250d458c4991a7d-02-F_MNXR94682_MNXM90191,NaN,([#6&v4:1](-[#8&v2:2]-[#1&v1:3])(-[#6&v4:4](-[...,2,Rule_47413
2,RR-02-c3681aa8011dc014-02-F_MNXR94689_MNXM101404,1.1.1.-,([#6&v4:1]-[#6&v4:2](=[#8&v2:3])-[#6&v4:4])>>(...,2,Rule_47414
3,RR-02-1364a3f2a297c78c-02-F_MNXR94689_MNXM9689,1.1.1.-,([#6&v4:1]-[#6&v4:2](-[#8&v2:3]-[#1&v1:4])(-[#...,2,Rule_47415
4,RR-02-2860703b5bba4808-02-F_MNXR94690_MNXM2313,NaN,([#8&v2:1](-[#8&v2:2]-[#6&v4:3])-[#1&v1:4])>>(...,2,Rule_47416


In [24]:
df_rr_cut['Reactants'] = 'Any'
df_rr_cut.head()

C:\Users\Duarte Velho\AppData\Local\Temp\ipykernel_2088\286152149.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_rr_cut['Reactants'] = 'Any'


,RuleID,EC_Numbers,SMARTS,Diameter,InternalID,Reactants
0,RR-02-fbdda75e23f518b6-02-F_MNXR94682_MNXM821,NaN,([#6&v4:1](=[#8&v2:2])(-[#6&v4:3](-[#6&v4:4])(...,2,Rule_47412,Any
1,RR-02-0250d458c4991a7d-02-F_MNXR94682_MNXM90191,NaN,([#6&v4:1](-[#8&v2:2]-[#1&v1:3])(-[#6&v4:4](-[...,2,Rule_47413,Any
2,RR-02-c3681aa8011dc014-02-F_MNXR94689_MNXM101404,1.1.1.-,([#6&v4:1]-[#6&v4:2](=[#8&v2:3])-[#6&v4:4])>>(...,2,Rule_47414,Any
3,RR-02-1364a3f2a297c78c-02-F_MNXR94689_MNXM9689,1.1.1.-,([#6&v4:1]-[#6&v4:2](-[#8&v2:3]-[#1&v1:4])(-[#...,2,Rule_47415,Any
4,RR-02-2860703b5bba4808-02-F_MNXR94690_MNXM2313,NaN,([#8&v2:1](-[#8&v2:2]-[#6&v4:3])-[#1&v1:4])>>(...,2,Rule_47416,Any


In [25]:
# Cria coluna Organisms no df_retro_rules, que é igual ao EC_Numbers
df_rr_cut['Organisms'] = df_rr_cut['EC_Numbers']
df_rr_cut.head()

C:\Users\Duarte Velho\AppData\Local\Temp\ipykernel_2088\3602570970.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_rr_cut['Organisms'] = df_rr_cut['EC_Numbers']


,RuleID,EC_Numbers,SMARTS,Diameter,InternalID,Reactants,Organisms
0,RR-02-fbdda75e23f518b6-02-F_MNXR94682_MNXM821,NaN,([#6&v4:1](=[#8&v2:2])(-[#6&v4:3](-[#6&v4:4])(...,2,Rule_47412,Any,NaN
1,RR-02-0250d458c4991a7d-02-F_MNXR94682_MNXM90191,NaN,([#6&v4:1](-[#8&v2:2]-[#1&v1:3])(-[#6&v4:4](-[...,2,Rule_47413,Any,NaN
2,RR-02-c3681aa8011dc014-02-F_MNXR94689_MNXM101404,1.1.1.-,([#6&v4:1]-[#6&v4:2](=[#8&v2:3])-[#6&v4:4])>>(...,2,Rule_47414,Any,1.1.1.-
3,RR-02-1364a3f2a297c78c-02-F_MNXR94689_MNXM9689,1.1.1.-,([#6&v4:1]-[#6&v4:2](-[#8&v2:3]-[#1&v1:4])(-[#...,2,Rule_47415,Any,1.1.1.-
4,RR-02-2860703b5bba4808-02-F_MNXR94690_MNXM2313,NaN,([#8&v2:1](-[#8&v2:2]-[#6&v4:3])-[#1&v1:4])>>(...,2,Rule_47416,Any,NaN


In [26]:
# merge dos dataframes por RuleID 
df_concatenated = pd.concat([df_react_rules_forward, df_rr_cut])
df_concatenated

,InternalID,RuleID,Reactants,SMARTS,EC_Numbers,Organisms,ReactionRuleSource,Diameter
0,Rule_0,"1,2-BenzoquinoneAdditionCyclization",Any,"[#7H2,SH1:11]-[#6:10]-[#6:9]-[#6:6]-1=[#6:1]-[...",spontaneous_reaction,spontaneous_reaction,ChemicalDamageMINE,None
1,Rule_1,"1,2-BenzoquinoneAlphaAddition_1,2-Benzoquinone",Any;O=C1C=CC=CC1=O,"[#6:10]-[#7H2,#16H1:9].[O:7]=[#6:3]-1-[#6;h1:4...",spontaneous_reaction,spontaneous_reaction,ChemicalDamageMINE,None
2,Rule_2,"1,2-BenzoquinoneAlphaAddition_Cystine",Any;N[C@@H](CS)C(=O)O,[O:7]=[#6:3]-1-[#6;h1:4]=[#6:5]-[#6:6]=[#6:1]-...,spontaneous_reaction,spontaneous_reaction,ChemicalDamageMINE,None
3,Rule_5,"1,2-BenzoquinoneBetaAddition_1,2Benzoquinone",Any;O=C1C=CC=CC1=O,"[#6:10]-[#7,#16;H1:9].[O:7]=[#6:3]-1-[#6:4]=[#...",spontaneous_reaction,spontaneous_reaction,ChemicalDamageMINE,None
4,Rule_6,"1,2-BenzoquinoneBetaAddition_Cystine",Any;N[C@@H](CS)C(=O)O,[O:7]=[#6:3]-1-[#6:4]=[#6;h1:5]-[#6:6]=[#6:1]-...,spontaneous_reaction,spontaneous_reaction,ChemicalDamageMINE,None
...,...,...,...,...,...,...,...,...
351698,Rule_282719,RR-02-f389fabfd88146dc-16-F_MNXR142602_MNXM3698,Any,([#8&v2:1](-[#6&v4:2](=[#8&v2:3])-[#6&v4:4](-[...,NaN,NaN,NaN,16
351699,Rule_282720,RR-02-b0c1129bdecc508e-16-F_MNXR142602_MNXM722899,Any,([#8&v2:1](-[#6&v4:2](=[#8&v2:3])-[#6&v4:4](-[...,NaN,NaN,NaN,16
351700,Rule_282721,RR-02-59f6dae94f362077-16-F_MNXR142603_MNXM13204,Any,([#8&v2:1](-[#6&v4:2]1:[#6&v4:3](-[#8&v2:4]-[#...,1.6.5.3,1.6.5.3,NaN,16
351701,Rule_282722,RR-02-4fccde049e5dee92-16-F_MNXR142603_MNXM89768,Any,([#8&v2:1]=[#6&v4:2]1-[#6&v4:3](-[#8&v2:4]-[#6...,1.6.5.3,1.6.5.3,NaN,16


In [27]:
df_concatenated.head()

,InternalID,RuleID,Reactants,SMARTS,EC_Numbers,Organisms,ReactionRuleSource,Diameter
0,Rule_0,"1,2-BenzoquinoneAdditionCyclization",Any,"[#7H2,SH1:11]-[#6:10]-[#6:9]-[#6:6]-1=[#6:1]-[...",spontaneous_reaction,spontaneous_reaction,ChemicalDamageMINE,None
1,Rule_1,"1,2-BenzoquinoneAlphaAddition_1,2-Benzoquinone",Any;O=C1C=CC=CC1=O,"[#6:10]-[#7H2,#16H1:9].[O:7]=[#6:3]-1-[#6;h1:4...",spontaneous_reaction,spontaneous_reaction,ChemicalDamageMINE,None
2,Rule_2,"1,2-BenzoquinoneAlphaAddition_Cystine",Any;N[C@@H](CS)C(=O)O,[O:7]=[#6:3]-1-[#6;h1:4]=[#6:5]-[#6:6]=[#6:1]-...,spontaneous_reaction,spontaneous_reaction,ChemicalDamageMINE,None
3,Rule_5,"1,2-BenzoquinoneBetaAddition_1,2Benzoquinone",Any;O=C1C=CC=CC1=O,"[#6:10]-[#7,#16;H1:9].[O:7]=[#6:3]-1-[#6:4]=[#...",spontaneous_reaction,spontaneous_reaction,ChemicalDamageMINE,None
4,Rule_6,"1,2-BenzoquinoneBetaAddition_Cystine",Any;N[C@@H](CS)C(=O)O,[O:7]=[#6:3]-1-[#6:4]=[#6;h1:5]-[#6:6]=[#6:1]-...,spontaneous_reaction,spontaneous_reaction,ChemicalDamageMINE,None


In [28]:
#save df_concatenated in a tsv file
#df_concatenated.to_csv('src/biocatalyzer/data/reactionrules/reaction_rules_biocatalyzer_concatenated.tsv', sep='\t', index=False)
#save df_concatenated in a bz2 file         
#df_concatenated.to_csv('src/biocatalyzer/data/reactionrules/reaction_rules_biocatalyzer_concatenated.tsv.bz2', sep='\t', index=False, compression='bz2')    

Renomeei  manualmente reaction_rules_biocatalyzer_concatenated.tsv para reaction_rules_biocatalyzer.tsv
depois corri com os dados do artigo (não com MS)

NOTA: correr como desenvolvedor da tool -> biocatalyzer_cli MOLENAME_SMILES_cleaned.tsv new_data --n_jobs=-1 (aqui não está como desenvolvedor)

# Identify 76 organisms from the paper and retrieve their respective KEGG IDs

In [29]:
a=82-6
a

76